# TerrariaGuide - 8B Fine-Tune
Run each cell in order. Make sure **Runtime > Change runtime type** is set to **T4 GPU** before starting.

In [ ]:
# Cell 1: Install dependencies
!pip install unsloth trl transformers datasets accelerate bitsandbytes peft sentencepiece -q

In [ ]:
# Cell 2: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Verify training file is accessible
import os
TRAINING_FILE = '/content/drive/MyDrive/TerrariaGuide/terraria_training_pairs.jsonl'
OUTPUT_DIR = '/content/drive/MyDrive/TerrariaGuide/models/terraria-guide-8b'

if os.path.exists(TRAINING_FILE):
    print(f'Training file found!')
    print(f'Lines: {sum(1 for _ in open(TRAINING_FILE))}')
else:
    print(f'ERROR: Training file not found at {TRAINING_FILE}')
    print('Please upload terraria_training_pairs.jsonl to your Drive at TerrariaGuide/terraria_training_pairs.jsonl')

In [ ]:
# Cell 3: Check GPU
import torch
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')
print(f'CUDA: {torch.version.cuda}')

In [ ]:
# Cell 4: Load model
from unsloth import FastLanguageModel

MAX_SEQ_LENGTH = 1024
MODEL_NAME = 'unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit'

print('Loading base model...')
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        'q_proj', 'k_proj', 'v_proj', 'o_proj',
        'gate_proj', 'up_proj', 'down_proj',
    ],
    lora_alpha=16,
    lora_dropout=0,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=42,
)
print('Model ready!')

In [ ]:
# Cell 5: Load dataset
from datasets import load_dataset

def format_prompt(example):
    return {
        'text': f"""### Instruction:\n{example['instruction']}\n\n### Response:\n{example['output']}<|end_of_text|>"""
    }

print('Loading dataset...')
dataset = load_dataset(
    'json',
    data_files=TRAINING_FILE,
    split='train'
)
dataset = dataset.map(format_prompt)
dataset = dataset.filter(lambda x: len(x['text']) < 4000)
print(f'Dataset size: {len(dataset)} pairs')

In [ ]:
# Cell 6: Train
from trl import SFTTrainer
from transformers import TrainingArguments
import os

os.makedirs(OUTPUT_DIR, exist_ok=True)

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    warmup_steps=50,
    num_train_epochs=1,
    learning_rate=2e-4,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=25,
    save_steps=500,
    save_total_limit=2,
    optim='adamw_8bit',
    weight_decay=0.01,
    lr_scheduler_type='cosine',
    seed=42,
    report_to='none',
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field='text',
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_num_proc=2,
    packing=False,
    args=training_args,
)

print('Starting training...')
trainer.train()

In [ ]:
# Cell 7: Save model to Drive
print('Saving model to Google Drive...')
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f'Model saved to {OUTPUT_DIR}')

In [ ]:
# Cell 8: Quick test
FastLanguageModel.for_inference(model)

def ask(question):
    prompt = f'### Instruction:\n{question}\n\n### Response:\n'
    inputs = tokenizer(prompt, return_tensors='pt').to('cuda')
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=512,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return response[len(prompt):].strip()

print('Test 1:', ask('What does the Eye of Cthulhu drop in Terraria?'))
print()
print('Test 2:', ask('How do I craft a Molten Pickaxe in Terraria?'))
print()
print('Test 3:', ask('What is the Wall of Flesh in Terraria?'))